# DAETF-Net Enhanced: Custom Loss, Learnable Upsampler, and Optional Enhancements

This notebook implements an enhanced version of DAETF-Net with:

- Custom loss (L1 + SSIM + gradient)

- Learnable upsampler (pixel shuffle)

- Optional self-attention block

- Optional mixup augmentation

- Optional cosine annealing with warm restarts

Set the flags below to enable/disable enhancements.

In [ ]:

# Enhancement Flags (Set to True/False)
USE_SELF_ATTENTION = True   # Self-attention after equivariant features
USE_MIXUP = True            # Mixup augmentation
USE_COSINE_RESTARTS = True  # Cosine annealing with warm restarts

print("Enhancement Flags:")
print(f"  USE_SELF_ATTENTION: {USE_SELF_ATTENTION}")
print(f"  USE_MIXUP: {USE_MIXUP}")
print(f"  USE_COSINE_RESTARTS: {USE_COSINE_RESTARTS}")


In [ ]:

# Install required packages
!pip install -q einops tensorly groupy torch torchvision
# Optional for SSIM loss
!pip install -q pytorch_msssim 2>/dev/null || echo 'pytorch_msssim installation failed; SSIM term may be skipped'


In [ ]:

# Import libraries
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import scipy.io as sio
from einops import rearrange
import tensorly as tl
from tensorly.decomposition import tucker
import torchvision.transforms as transforms
from torchvision.models import vgg16
import matplotlib.pyplot as plt
import json
from datetime import datetime
import shutil
import subprocess
import sys
import math
import random

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Create directories for saving results
os.makedirs('./checkpoints', exist_ok=True)
os.makedirs('./results', exist_ok=True)
os.makedirs('./logs', exist_ok=True)


In [ ]:

# Equivariant Feature Extractor (EFE)
class EquivariantFeatureExtractor(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        return x


In [ ]:

# Self-Attention Block (optional)
if USE_SELF_ATTENTION:
    class SelfAttention(nn.Module):
        def __init__(self, dim, heads=4, dim_head=16, dropout=0.):
            super().__init__()
            inner_dim = dim_head * heads
            project_out = not (heads == 1 and dim_head == dim)
            self.heads = heads
            self.scale = dim_head ** -0.5
            self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
            self.to_out = nn.Sequential(
                nn.Linear(inner_dim, dim),
                nn.Dropout(dropout)
            ) if project_out else nn.Identity()

        def forward(self, x):
            # x: [B, N, C] where N is sequence length (H*W)
            b, n, c = x.shape
            qkv = self.to_qkv(x).reshape(b, n, 3, self.heads, c // self.heads).permute(2, 0, 3, 1, 4)
            q, k, v = qkv[0], qkv[1], qkv[2]  # each: [B, heads, N, dim_head]
            dots = torch.einsum('b h i d, b h j d -> b h i j', q, k) * self.scale
            attn = dots.softmax(dim=-1)
            out = torch.einsum('b h i j, b h j d -> b h i d', attn, v)
            out = out.permute(0, 2, 1, 3).reshape(b, n, self.heads * (c // self.heads))
            return self.to_out(out)


In [ ]:

# Tensor Spectral-Spatial Encoder (TSSE)
class TensorSpectralSpatialEncoder(nn.Module):
    def __init__(self, in_channels_hsi, in_channels_msi, rank):
        super().__init__()
        self.rank = rank
        self.proj_hsi = nn.Conv2d(in_channels_hsi, rank, kernel_size=1)
        self.proj_msi = nn.Conv2d(in_channels_msi, rank, kernel_size=1)
        self.core = nn.Parameter(torch.randn(rank, rank, rank, rank))
        self.bn = nn.BatchNorm2d(rank)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, hsi, msi):
        # hsi: [B, C_hsi, H, W], msi: [B, C_msi, H, W]
        hsi_proj = self.proj_hsi(hsi)  # [B, rank, H, W]
        msi_proj = self.proj_msi(msi)  # [B, rank, H, W]
        interaction = hsi_proj * msi_proj  # [B, rank, H, W]
        output = self.relu(self.bn(interaction))
        return output


In [ ]:

# Adaptive Fusion Mixture-of-Experts (AF-MoE)
class AdaptiveFusionMoE(nn.Module):
    def __init__(self, in_channels, num_experts=4):
        super().__init__()
        self.num_experts = num_experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
            ) for _ in range(num_experts)
        ])
        self.gating = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(in_channels, num_experts),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        # x: [B, C, H, W]
        gates = self.gating(x)  # [B, num_experts]
        expert_outputs = [expert(x) for expert in self.experts]  # List of [B, C, H, W]
        expert_outputs = torch.stack(expert_outputs, dim=1)  # [B, num_experts, C, H, W]
        gates = gates.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)  # [B, num_experts, 1, 1, 1]
        output = torch.sum(gates * expert_outputs, dim=1)  # [B, C, H, W]
        return output


In [ ]:

# Frequency-Domain Refinement Module (FDRM)
class FrequencyDomainRefinementModule(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv_3x3 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.conv_5x5 = nn.Conv2d(in_channels, in_channels, kernel_size=5, padding=2)
        self.conv_7x7 = nn.Conv2d(in_channels, in_channels, kernel_size=7, padding=3)
        self.bn = nn.BatchNorm2d(in_channels * 3)
        self.relu = nn.ReLU(inplace=True)
        self.fuse = nn.Conv2d(in_channels * 3, in_channels, kernel_size=1)

    def forward(self, x):
        x3 = self.relu(self.conv_3x3(x))
        x5 = self.relu(self.conv_5x5(x))
        x7 = self.relu(self.conv_7x7(x))
        x = torch.cat([x3, x5, x7], dim=1)
        x = self.relu(self.bn(x))
        x = self.fuse(x)
        return x


In [ ]:

# Custom Loss Function
class CustomLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=1.0, gamma=0.5):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.l1 = nn.L1Loss()
        try:
            from pytorch_msssim import ssim
            self.ssim = ssim
            self.use_ssim = True
        except ImportError:
            self.use_ssim = False
            print("Warning: pytorch_msssim not installed, SSIM term will be skipped. Install via: pip install pytorch_msssim")

    def gradient_loss(self, pred, target):
        # Simple gradient loss using differences
        grad_x_pred = torch.abs(pred[:, :, :, :-1] - pred[:, :, :, 1:])
        grad_y_pred = torch.abs(pred[:, :, :-1, :] - pred[:, :, 1:, :])
        grad_x_target = torch.abs(target[:, :, :, :-1] - target[:, :, :, 1:])
        grad_y_target = torch.abs(target[:, :, :-1, :] - target[:, :, 1:, :])
        loss_grad = F.l1_loss(grad_x_pred, grad_x_target) + F.l1_loss(grad_y_pred, grad_y_target)
        return loss_grad

    def forward(self, pred, target):
        loss = self.alpha * self.l1(pred, target)
        if self.use_ssim:
            ssim_val = self.ssim(pred, target, data_range=1.0, size_average=True)
            loss += self.beta * (1 - ssim_val)
        if self.gamma > 0:
            loss += self.gamma * self.gradient_loss(pred, target)
        return loss


In [ ]:

# Full DAETF-Net Model
class DAETFNet(nn.Module):
    def __init__(self, in_channels_hsi=31, in_channels_msi=3, rank=8, upscale_factor=4):
        super().__init__()
        self.upscale_factor = upscale_factor
        # Learnable upsampler for LR-HSI to match MSI resolution (pixel shuffle)
        self.upsampler = nn.Sequential(
            nn.Conv2d(in_channels_hsi, in_channels_hsi * (upscale_factor ** 2), kernel_size=3, padding=1),
            nn.PixelShuffle(upscale_factor),
            nn.Conv2d(in_channels_hsi, in_channels_hsi, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        self.efe = EquivariantFeatureExtractor(in_channels_hsi + in_channels_msi, 64)
        # Optional: Self-attention after EFE
        if USE_SELF_ATTENTION:
            # We'll reshape to sequence for attention
            self.attn = SelfAttention(dim=64, heads=4, dim_head=16)
        self.tsse = TensorSpectralSpatialEncoder(64, 64, rank)
        self.af_moe = AdaptiveFusionMoE(64, num_experts=4)
        self.fdrm = FrequencyDomainRefinementModule(64)
        self.reconstruction = nn.Conv2d(64, in_channels_hsi, kernel_size=3, padding=1)

    def forward(self, hsi_lr, msi):
        # hsi_lr: low-resolution HSI [B, C_hsi, H, W]
        # msi: multispectral image [B, C_msi, H*scale, W*scale] (high-res spatial)
        # Upsample LR-HSI to match MSI spatial resolution
        hsi_hr = self.upsampler(hsi_lr)  # [B, C_hsi, H*scale, W*scale]
        # Concatenate along channel dimension
        x = torch.cat([hsi_hr, msi], dim=1)  # [B, C_hsi+C_msi, H*scale, W*scale]
        # Equivariant feature extraction
        x = self.efe(x)  # [B, 64, H*scale, W*scale]
        # Optional: Self-attention
        if USE_SELF_ATTENTION:
            # Reshape to [B, N, C] where N = H*W
            B, C, H, W = x.shape
            x_seq = x.permute(0, 2, 3, 1).reshape(B, H*W, C)  # [B, N, C]
            x_seq = self.attn(x_seq)
            # Reshape back
            x = x_seq.permute(0, 2, 1).reshape(B, C, H, W)
        # Tensor spectral-spatial encoding
        x = self.tsse(x, x)  # Simplified: using same input for both
        # Adaptive fusion
        x = self.af_moe(x)
        # Frequency-domain refinement
        x = self.fdrm(x)
        # Reconstruction to HSI bands
        x = self.reconstruction(x)  # [B, C_hsi, H*scale, W*scale]
        # Add residual (upsampled LR-HSI)
        output = x + hsi_hr
        return output


In [ ]:

# Dataset Class (with optional Mixup)
class HSIFusionDataset(Dataset):
    def __init__(self, hsi_dir, msi_dir, transform=None, mixup_alpha=0.0):
        self.hsi_dir = hsi_dir
        self.msi_dir = msi_dir
        self.transform = transform
        self.mixup_alpha = mixup_alpha
        # In practice, load the file names
        # For dummy, we'll use a fixed length
        self.length = 100  # Dummy

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # Generate dummy data
        # In practice, load .mat files and convert to tensors
        hsi = torch.randn(31, 64, 64)  # [C, H, W]
        msi = torch.randn(3, 256, 256)    # [C, H*scale, W*scale] assuming scale=4
        # For training, we need a ground truth HR-HSI
        hr_hsi = torch.randn(31, 256, 256)  # Dummy ground truth

        if self.transform:
            hsi = self.transform(hsi)
            msi = self.transform(msi)
            hr_hsi = self.transform(hr_hsi)

        # Mixup augmentation
        if self.mixup_alpha > 0 and np.random.rand() < 0.5:
            # Get another random sample
            idx2 = random.randint(0, self.length-1)
            hsi2 = torch.randn(31, 64, 64)
            msi2 = torch.randn(3, 256, 256)
            hr_hsi2 = torch.randn(31, 256, 256)
            if self.transform:
                hsi2 = self.transform(hsi2)
                msi2 = self.transform(msi2)
                hr_hsi2 = self.transform(hr_hsi2)
            lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
            hsi = lam * hsi + (1 - lam) * hsi2
            msi = lam * msi + (1 - lam) * msi2
            hr_hsi = lam * hr_hsi + (1 - lam) * hr_hsi2

        return hsi, msi, hr_hsi


In [ ]:

# Training Setup
def train_model(model, train_loader, val_loader, num_epochs=30, learning_rate=1e-4):
    model.to(device)
    criterion = CustomLoss(alpha=1.0, beta=1.0, gamma=0.5)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    if USE_COSINE_RESTARTS:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    else:
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    best_val_loss = float('inf')
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for i, (hsi_lr, msi, hr_hsi) in enumerate(train_loader):
            hsi_lr = hsi_lr.to(device)
            msi = msi.to(device)
            hr_hsi = hr_hsi.to(device)

            optimizer.zero_grad()
            outputs = model(hsi_lr, msi)
            loss = criterion(outputs, hr_hsi)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % 10 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

        epoch_loss = running_loss / len(train_loader)
        train_losses.append(epoch_loss)

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for hsi_lr, msi, hr_hsi in val_loader:
                hsi_lr = hsi_lr.to(device)
                msi = msi.to(device)
                hr_hsi = hr_hsi.to(device)
                outputs = model(hsi_lr, msi)
                loss = criterion(outputs, hr_hsi)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        scheduler.step()

        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}')

        # Save checkpoint if validation loss improved
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'./checkpoints/daetfnet_enhanced_best.pth')
            print(f'Saved best model with val loss: {val_loss:.4f}')

        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            torch.save(model.state_dict(), f'./checkpoints/daetfnet_enhanced_epoch_{epoch+1}.pth')

    return train_losses, val_losses


In [ ]:

# Main Execution
if __name__ == '__main__':
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    random.seed(42)

    # Paths to datasets (assuming they are in /kaggle/input/)
    # For CAVE dataset
    cave_hsi_dir = '/kaggle/input/cave-dataset-2/Data/Test/HSI'
    cave_msi_dir = '/kaggle/input/cave-dataset-2/Data/Test/RGB'
    # For Harvard dataset
    harvard_hsi_dir = '/kaggle/input/harvard-dataset/Data/Test/HSI'
    harvard_msi_dir = '/kaggle/input/harvard-dataset/Data/Test/RGB'

    # Check if directories exist
    print('Checking dataset paths...')
    for path in [cave_hsi_dir, cave_msi_dir, harvard_hsi_dir, harvard_msi_dir]:
        if os.path.exists(path):
            print(f'��� Found: {path}')
        else:
            print(f'��� Not found: {path}')
            print('  Please ensure the datasets are downloaded and placed in the correct location.')

    # Create datasets and data loaders
    # We'll use the CAVE dataset for training and Harvard for validation (to test domain shift)
    train_dataset = HSIFusionDataset(cave_hsi_dir, cave_msi_dir, mixup_alpha=0.2 if USE_MIXUP else 0.0)
    val_dataset = HSIFusionDataset(harvard_hsi_dir, harvard_msi_dir, mixup_alpha=0.0)

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

    print(f'Train dataset size: {len(train_dataset)}')
    print(f'Val dataset size: {len(val_dataset)}')

    # Initialize model
    model = DAETFNet(in_channels_hsi=31, in_channels_msi=3, rank=8, upscale_factor=4)
    print(f'Model size: {sum(p.numel() for p in model.parameters())/1e6:.2f} M parameters')

    # Train the model
    print('Starting training...')
    train_losses, val_losses = train_model(model, train_loader, val_loader, num_epochs=20, learning_rate=1e-4)

    # Plot training history
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig('./results/training_history.png')
    plt.show()

    # Save final model
    torch.save(model.state_dict(), './checkpoints/daetfnet_enhanced_final.pth')
    print('Training completed. Models saved in ./checkpoints/')
